In [1]:
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.random.seed(42)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


Python: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
Pandas: 2.3.3
NumPy: 2.3.5


In [2]:
PROJECT_ROOT = Path(r"D:\STAT3013.Q12_Group01")  
RAW      = PROJECT_ROOT / "data" / "raw"
FEATURES = PROJECT_ROOT / "features"
RESULTS  = PROJECT_ROOT / "results"
FIGURES  = PROJECT_ROOT / "figures"

for p in [FEATURES, RESULTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW exists:", RAW.exists())
print("RAW files:", [f.name for f in RAW.glob("*")])


PROJECT_ROOT: D:\STAT3013.Q12_Group01
RAW exists: True
RAW files: ['campaign_desc.csv', 'campaign_table.csv', 'causal_data.csv', 'coupon.csv', 'coupon_redempt.csv', 'hh_demographic.csv', 'product.csv', 'transaction_data.csv']


In [3]:
def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names to lowercase, strip spaces."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df

def safe_read_csv(path, **kwargs):
    """Read csv with a small safety net for encoding issues."""
    try:
        return pd.read_csv(path, **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1", **kwargs)

def mape(y_true, y_pred):
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    denom = np.where(y_true == 0, 1, y_true)
    return np.mean(np.abs((y_true - y_pred) / denom))


In [4]:
# load 3 file core để test
tx_path      = RAW / "transaction_data.csv"
product_path = RAW / "product.csv"
causal_path  = RAW / "causal_data.csv"

tx_raw      = safe_read_csv(tx_path)
product_raw = safe_read_csv(product_path)
causal_raw  = safe_read_csv(causal_path)

print("Shapes:", tx_raw.shape, product_raw.shape, causal_raw.shape)

tx = lower_cols(tx_raw)
product = lower_cols(product_raw)
causal_data = lower_cols(causal_raw)

print("\nTX columns:", tx.columns.tolist()[:15])
print("PRODUCT columns:", product.columns.tolist()[:15])
print("CAUSAL columns:", causal_data.columns.tolist()[:15])

tx.head()


Shapes: (1048575, 12) (92353, 7) (1048575, 5)

TX columns: ['household_key', 'basket_id', 'day', 'product_id', 'quantity', 'sales_value', 'store_id', 'retail_disc', 'trans_time', 'week_no', 'coupon_disc', 'coupon_match_disc']
PRODUCT columns: ['product_id', 'manufacturer', 'department', 'brand', 'commodity_desc', 'sub_commodity_desc', 'curr_size_of_product']
CAUSAL columns: ['product_id', 'store_id', 'week_no', 'display', 'mailer']


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [5]:
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed or error:", e)

try:
    import pytorch_lightning as pl
    import pytorch_forecasting as pf
    print("Lightning:", pl.__version__)
    print("PyTorch Forecasting:", pf.__version__)
except Exception as e:
    print("Lightning/Forecasting not installed or error:", e)



Torch: 2.9.1+cpu
CUDA available: False


Lightning: 2.5.6
PyTorch Forecasting: 1.5.0


In [6]:
config = {
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "RAW": str(RAW),
    "FEATURES": str(FEATURES),
    "RESULTS": str(RESULTS),
    "FIGURES": str(FIGURES),
    "SEED": 42
}

with open(PROJECT_ROOT / "src" / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved config.json to src/")


Saved config.json to src/
